# Equation of State Calculation

In [27]:
import jax.numpy as jnp
from jax import random, jit
from matplotlib import pyplot as plt
import numpy as np
from jax_md import space, energy, simulate, quantity

In [ ]:
# Simulation parameters
num_steps = 10000
num_steps_equilibrium = 2000
dt = 1e-3
sigma = 1.0
epsilon = 1.0
Nx = 10
dim = 3
spacing = 1.05 * sigma
side_length = Nx * spacing
num_particles = Nx**dim


In [ ]:
# Periodic space
displacement, shift = space.periodic(side_length)
energy_fn = energy.lennard_jones_pair(displacement, sigma=sigma, epsilon=epsilon, r_cutoff=2.5 * sigma)
energy_fn = jit(energy_fn)


In [ ]:
# Function to compute average pressure for given kT and density
def compute_avg_pressure(kT, density):
    box_size = (num_particles / density) ** (1 / dim)
    init_fn, apply_fn = simulate.nvt_langevin(energy_fn, shift, dt=dt, kT=kT)
    apply_fn = jit(apply_fn)
    
    key = random.PRNGKey(0)
    base_R = np.stack([np.array(r) for r in np.ndindex((Nx,)*dim)]) * spacing
    max_offset = (spacing - sigma) / 2
    random_offset = np.random.uniform(-max_offset, max_offset, size=(num_particles, dim))
    R = jnp.array(base_R + random_offset)
    state = init_fn(key, R)
    
    pressures = jnp.zeros(num_steps_equilibrium)
    
    for t in range(num_steps):
        state = apply_fn(state)
        if t >= (num_steps - num_steps_equilibrium):
            pressure_t = quantity.pressure(energy_fn, state.position, box=box_size, kT=kT)
            pressures = pressures.at[t - (num_steps - num_steps_equilibrium)].set(pressure_t)
    
    return jnp.mean(pressures)

In [ ]:
# Vary temperature (kT)
kT_values = jnp.linspace(0.5, 2.5, 5)  # Example range of kT values
pressures_vs_kT = jnp.array([compute_avg_pressure(kT, density=num_particles / side_length**dim) for kT in kT_values])

plt.figure()
plt.plot(kT_values, pressures_vs_kT, 'o-', label='Pressure vs kT')
plt.xlabel('Temperature (kT)')
plt.ylabel('Averaged Pressure')
plt.legend()
plt.show()

In [ ]:
# Vary density
density_values = jnp.linspace(0.1, 1.0, 5) * (num_particles / side_length**dim)  # Example densities
pressures_vs_density = jnp.array([compute_avg_pressure(kT=1.0, density=d) for d in density_values])

plt.figure()
plt.plot(density_values, pressures_vs_density, 'o-', label='Pressure vs Density')
plt.xlabel('Density')
plt.ylabel('Averaged Pressure')
plt.legend()
plt.show()

